In [ ]:
names = open('names.txt', 'r').read().splitlines()
len(names)

In [ ]:
chars = sorted(list(set(''.join(names))))

c_to_i = {c:i+1 for i,c in enumerate(chars)}
c_to_i['.'] = 0

i_to_c = {i:c for c,i in c_to_i.items()}

Bigram Probability Model

In [ ]:
import torch

N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
for name in names:
    chs = ['.'] + list(name) + ['.']
    for bigram in zip(chs, chs[1:]):
        ix1 = c_to_i[bigram[0]]
        ix2 = c_to_i[bigram[1]]

        N[ix1, ix2] += 1


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = i_to_c[i] + i_to_c[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
P = N.float()
P /= P.sum(1, keepdim=True)

In [ ]:
g = torch.Generator().manual_seed(2147483647)


for i in range(20):
    ix = 0
    out = []

    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        
        out.append(i_to_c[ix])

        if ix == 0:
            break

    print(''.join(out[:-1]))

In [ ]:
log_likelihood = 0.0
n = 0

for name in names:
    chs = ['.'] + list(name) + ['.']
    for bigram in zip(chs, chs[1:]):
        ix1 = c_to_i[bigram[0]]
        ix2 = c_to_i[bigram[1]]

        prob = P[ix1, ix2]
        log_prob = torch.log(prob)
        log_likelihood += log_prob
        n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood

print(f'{nll=}')
print(f'{nll/n=}')

Bigram Model using random weights and back propagation

In [ ]:
xs, ys = [], []

for name in names:
    chs = ['.'] + list(name) + ['.']
    for bigram in zip(chs, chs[1:]):
        ix1 = c_to_i[bigram[0]]
        ix2 = c_to_i[bigram[1]]

        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

print(f'number of examples: {num}')

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
import torch.nn.functional as F

for k in range(200):

    # Forward Prop
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())

    # Backwards Prop
    W.grad = None
    loss.backward()

    # Update Step
    W.data += -50 * W.grad



In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(10):

    out = []
    ix = 0

    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)
        
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(i_to_c[ix])

        if ix == 0:
            break
            
    print(''.join(out))
        